# 6.6.6 color-code decoupling

This notebook reads the shared paper example and the canonical reproduction output. Run the corresponding script before executing the notebook.

In [1]:
from pathlib import Path
import json
import sys
from sage.all import Matrix
SOURCE_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'decoupling' / '__init__.py').is_file()
)
sys.path.insert(0, str(SOURCE_ROOT))
for module_name in tuple(sys.modules):
    if module_name == 'decoupling' or module_name.startswith('decoupling.'):
        del sys.modules[module_name]
from decoupling.paper_examples import COLOR_CODE_666
from decoupling import R, x, y
from sage.misc.sage_eval import sage_eval

def matrix_from_strings(rows):
    return Matrix(
        R,
        [[R(sage_eval(entry, locals={'x': x, 'y': y})) for entry in row] for row in rows],
    )

case = COLOR_CODE_666
result_path = SOURCE_ROOT / 'results' / 'decoupling' / 'color_code_666.json'
payload = json.loads(result_path.read_text(encoding='utf-8'))
record = payload['result']
assert record['verification_status'] == 'passed'
assert record['superlattice_basis'] == [list(vector) for vector in case.superlattice_basis]
record['actual_q'], record['actual_p_x'], record['actual_p_z'], record['actual_t']

(6, 1, 1, 2)

## Paper input and paper-specified superlattice

In [2]:
input_excitation_map = case.excitation_map()
input_excitation_map, case.superlattice_basis, record['actual_square_period']

(
[         x*y + x + 1          x*y + y + 1                    0                    0]
[                   0                    0 1 + y^-1 + x^-1*y^-1 1 + x^-1 + x^-1*y^-1],

((3, 0), (2, 1)), 3
)

## Computed standard complex

In [3]:
h_x_tilde = matrix_from_strings(record['h_x_tilde'])
h_z_tilde_dagger = matrix_from_strings(record['h_z_tilde_dagger'])
h_x_tilde, h_z_tilde_dagger

(
                                       [    0     0     0]
                                       [    1     0     0]
                                       [    0 y + 1     0]
[    1     0     0     0     0     0]  [    0 x + 1     0]
[    0     0 x + 1 y + 1     0     0]  [    0     0 y + 1]
[    0     0     0     0 x + 1 y + 1], [    0     0 x + 1]
)

## Computed inverse chain isomorphisms

In [4]:
psi_2_inverse = matrix_from_strings(record['psi_2_inverse'])
psi_1_inverse = matrix_from_strings(record['psi_1_inverse'])
psi_0_inverse = matrix_from_strings(record['psi_0_inverse'])
psi_2_inverse, psi_1_inverse, psi_0_inverse

(
                     [             1              1              x          y + 1              0              1]
                     [             1         x^-1*y              1         x^-1*y              x y + 1 + x^-1*y]
                     [             0         x^-1*y              0              0              1              0]
[    1     1 x + 1]  [             0              1              x          y + 1              x          y + 1]
[    0     1     1]  [             1              1              0              0              x              y]
[    0     0     1], [             0         x^-1*y              1         x^-1*y              0         x^-1*y],

[1 0 0]
[1 1 0]
[1 0 1]
)

The chain isomorphisms are non-unique. Different valid Gröbner bases, quotient bases, and pivot choices can produce matrices that do not match the paper entry by entry. The reproduction script therefore verifies the forward and inverse chain equations, mutual invertibility, and symplectic/QCA decoupling identities.

In [5]:
record['actual_psi_1_inverse_degree'], record['decoupling_runtime_seconds'], record['verification_status']

(2, 0.03560266600106843, 'passed')